# TrashTrack — EDA (TACO)
MSDS 498 Capstone · Group 4.

This notebook profiles the **TACO** annotations to inform modeling choices. The two questions that drive the whole detector design:
1. **How imbalanced are the classes?** (affects loss/sampling)
2. **How small are the objects?** (affects input resolution / tiling)

It runs on the annotation JSON alone (no image download needed). To extend to **UAVVaste** / **RoLID-11K**, point `ANN_PATH` at their COCO-format annotations.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ANN_PATH = '../data/taco_annotations.json'  # swap for UAVVaste / RoLID-11K
# for Google Colab, uncomment the following lines to upload the annotation file from your local machine
# from google.colab import files
# uploaded = files.upload()  # pick taco_annotations.json from your machine
# ANN_PATH = 'taco_annotations.json'

d = json.load(open(ANN_PATH))
images = pd.DataFrame(d['images'])
anns   = pd.DataFrame(d['annotations'])
cats   = pd.DataFrame(d['categories'])
print(len(images), 'images |', len(anns), 'annotations |', len(cats), 'categories')

## 1. Enrich annotations with class, supercategory, and size features

In [ ]:
cat_map = dict(zip(cats.id, cats.name)); super_map = dict(zip(cats.id, cats.supercategory))
anns['cat_name'] = anns['category_id'].map(cat_map)
anns['supercat'] = anns['category_id'].map(super_map)
bb = np.array(anns['bbox'].tolist(), float)
anns['bw'], anns['bh'] = bb[:,2], bb[:,3]
anns['bbox_area'] = anns['bw']*anns['bh']
dims = images.set_index('id')[['width','height']]
anns = anns.join(dims, on='image_id')
anns['rel_area_pct'] = 100*anns['bbox_area']/(anns['width']*anns['height'])
def bucket(a):
    return 'small' if a<32**2 else ('medium' if a<96**2 else 'large')
anns['size_bucket'] = anns['bbox_area'].apply(bucket)
anns[['cat_name','supercat','bbox_area','rel_area_pct','size_bucket']].head()

## 2. Class imbalance

In [ ]:
cc = anns['cat_name'].value_counts()
print('imbalance ratio (max:min) =', round(cc.max()/cc.min()))
print('classes with <20 instances:', int((cc<20).sum()), 'of', len(cc))
print('top-5 share:', round(cc.head(5).sum()/cc.sum()*100,1), '%')
cc.head(20)[::-1].plot.barh(figsize=(7,6), color='#0e7c86')
plt.title('TACO — top 20 classes'); plt.xlabel('count'); plt.tight_layout(); plt.show()

## 3. Small-object analysis
The single most important modeling fact for this project.

In [ ]:
print('median bbox as % of image:', round(anns.rel_area_pct.median(),3), '%')
print('share < 1% of image  :', round((anns.rel_area_pct<1).mean()*100,1), '%')
print(anns.size_bucket.value_counts())
plt.hist(np.clip(anns.rel_area_pct,0,5), bins=60, color='#e4572e')
plt.axvline(1, ls='--', color='k', label='1% of image'); plt.legend()
plt.title('bbox area as % of image (clipped 5%)'); plt.xlabel('%'); plt.show()

## 4. Objects per image & resolution

In [ ]:
per = anns.groupby('image_id').size()
print('objects/image  mean %.2f  median %d  max %d' % (per.mean(), per.median(), per.max()))
print('image width  median', int(images.width.median()), '| height median', int(images.height.median()))
per.clip(upper=20).plot.hist(bins=20, color='#0e7c86')
plt.title('litter objects per image'); plt.show()

## 5. Takeaways for modeling
- **Severe imbalance** → use focal loss / class-balanced sampling; consider collapsing 60 classes to a single `litter` class for the first fine-tune, then refine.
- **Tiny objects** (median ~0.35% of image, 67% under 1%) → train at high input resolution and/or **tile** large images; small anchors.
- **Very large source images** (median ~2448×3264) → tiling is not optional for TACO.
- Re-run this notebook on UAVVaste (aerial → even smaller objects) and RoLID-11K (dashcam → motion blur) to compare the size distributions before benchmarking.